# Issue [L1]: Standard Logit Lens (GPT-2 & GePpeTto)

This notebook develops and validates the logic for the **Standard Logit Lens** (nostalgebraist, 2020) before porting it to the official module `src/ws4_lens/`.

### Contract Specifications (`docs/CONTRACTS.md`):
- For $l < L$: $\text{logits}_l = W_U(\text{LN}_f(h_l))$
- For $l = L$: $\text{logits}_L = W_U(h_L)$ (in Hugging Face, $h_L$ already has the final layer normalization applied)
- **Sanity Check**: $\text{logits}_L$ must be numerically identical to `model(input_ids).logits`.

## 1. Environment Setup and Imports

In [1]:
import sys
from pathlib import Path
import torch

# Add repository 'src/' directory to sys.path to enable project imports
ROOT_DIR = Path.cwd().resolve().parent
SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lib.data import load_model
from lib.registry import get_model_spec

print(f"Root dir: {ROOT_DIR}")
print(f"PyTorch version: {torch.__version__}")

/Users/matteosalvini/Desktop/Magistrale/HLT/HLT_Project-LLMs_on_Probing_and_Logit_Lens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Root dir: /Users/matteosalvini/Desktop/Magistrale/HLT/HLT_Project-LLMs_on_Probing_and_Logit_Lens
PyTorch version: 2.15.0


## 2. Loading GPT-2
Load GPT-2 using the shared project interface `lib.data.load_model`.

In [2]:
model_spec = get_model_spec("gpt2")
print("Model spec:", model_spec)

model, tokenizer, metadata = load_model(
    checkpoint=model_spec["hf_id"],
    kind=model_spec["kind"],
    tokenizer_kwargs={"add_prefix_space": True},
)

print("\nModel loaded successfully:")
print(f"- Device: {metadata['device']}")
print(f"- Layers (L): {metadata['num_layers']}")
print(f"- Hidden size (d): {metadata['hidden_size']}")
print(f"- Vocab size (V): {metadata['vocab_size']}")

# Inspection of key Logit Lens components in GPT2LMHeadModel:
print("\nKey Logit Lens components:")
print("- ln_f:", model.transformer.ln_f)
print("- lm_head:", model.lm_head)

Model spec: {'key': 'gpt2', 'hf_id': 'gpt2', 'kind': 'decoder', 'lang': 'en', 'num_layers': 12, 'hidden_size': 768, 'dtype': 'float32', 'revision': 'main'}


W1114 20:02:30.101000 39073 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9003.00it/s]



Model loaded successfully:
- Device: mps
- Layers (L): 12
- Hidden size (d): 768
- Vocab size (V): 50257

Key Logit Lens components:
- ln_f: LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
- lm_head: Linear(in_features=768, out_features=50257, bias=False)


## 3. Implementation of `run_model`

Signature from `docs/CONTRACTS.md`:
```python
def run_model(model, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> list[torch.Tensor]
```
Executes the causal model in `eval` mode with `torch.no_grad()`, returning a list of $L + 1$ tensors of shape `[B, T, d]`.

In [3]:
def run_model(
    model: torch.nn.Module,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor | None = None,
) -> list[torch.Tensor]:
    """
    Run the causal model and extract all intermediate hidden states.

    Parameters
    ----------
    model : torch.nn.Module
        Pretrained causal LM (e.g. GPT2LMHeadModel).
    input_ids : torch.Tensor
        Token index tensor of shape [B, T].
    attention_mask : torch.Tensor | None
        Attention mask of shape [B, T]. If None, defaults to all ones.

    Returns
    -------
    list[torch.Tensor]
        List of L + 1 tensors of shape [B, T, d], from layer 0 (embeddings)
        up to layer L.
    """
    model.eval()
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = getattr(model, "device", input_ids.device)
    input_ids = input_ids.to(device)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

    # outputs.hidden_states is a tuple of L + 1 tensors of shape [B, T, d]
    return list(outputs.hidden_states)

## 4. Implementation of `logit_lens`

Signature from `docs/CONTRACTS.md`:
```python
def logit_lens(model, hidden_states: list[torch.Tensor]) -> list[torch.Tensor]
```
- For $l < L$: project intermediate hidden state through final LayerNorm $\text{LN}_f$ and unembedding head $W_U$.
- For $l = L$: apply unembedding head $W_U(h_L)$ directly (since $h_L$ in Hugging Face is already post-LN_f).
- Returns $L + 1$ tensors of shape `[B, T, V]`.

In [4]:
def _get_lens_components(model: torch.nn.Module) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Extract final layer norm (ln_f) and unembedding head (lm_head)."""
    # Final layer normalization (ln_f)
    if hasattr(model, "transformer") and hasattr(model.transformer, "ln_f"):
        ln_f = model.transformer.ln_f
    elif hasattr(model, "ln_f"):
        ln_f = model.ln_f
    else:
        raise AttributeError("Could not find final layer normalization module (ln_f).")

    # Unembedding projection head (lm_head)
    if hasattr(model, "lm_head"):
        lm_head = model.lm_head
    elif hasattr(model, "get_output_embeddings"):
        lm_head = model.get_output_embeddings()
    else:
        raise AttributeError("Could not find output unembedding layer (lm_head).")

    return ln_f, lm_head


def logit_lens(model: torch.nn.Module, hidden_states: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    Compute Standard Logit Lens logits across all layers from 0 to L.

    Parameters
    ----------
    model : torch.nn.Module
        Causal LM containing ln_f and lm_head.
    hidden_states : list[torch.Tensor]
        List of L + 1 tensors of shape [B, T, d].

    Returns
    -------
    list[torch.Tensor]
        List of L + 1 tensors of shape [B, T, V].
    """
    ln_f, lm_head = _get_lens_components(model)
    num_layers = len(hidden_states) - 1
    logits_list: list[torch.Tensor] = []

    for l, h_l in enumerate(hidden_states):
        if l < num_layers:
            # Intermediate layers: apply ln_f followed by lm_head
            logits_l = lm_head(ln_f(h_l))
        else:
            # Final layer L: h_L in Hugging Face is already normalized by ln_f
            logits_l = lm_head(h_l)
        logits_list.append(logits_l)

    return logits_list

## 5. Verification & Sanity Check with GPT-2

For a test prompt, we verify:
1. Shapes of hidden states are $[B, T, d]$.
2. Shapes of lens logits are $[B, T, V]$.
3. **Mandatory Sanity Check**: $\text{logits}_L$ strictly matches `model(input_ids).logits`.

In [5]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print(f"Prompt: '{prompt}'")
print(f"Token IDs shape: {input_ids.shape}")

# 1. Run model
hidden_states = run_model(model, input_ids, attention_mask)
print(f"Extracted layers count: {len(hidden_states)} (L + 1 = 13 for GPT-2)")
print(f"Shape of each layer hidden state: {hidden_states[0].shape}")

# 2. Run logit lens
lens_logits = logit_lens(model, hidden_states)
print(f"Computed lens logits count: {len(lens_logits)}")
print(f"Shape of each lens logit: {lens_logits[0].shape} (B, T, V)")

# 3. Official sanity check: Layer L vs Model Logits
with torch.no_grad():
    official_logits = model(input_ids.to(model.device), attention_mask=attention_mask.to(model.device)).logits

logits_L = lens_logits[-1]
diff = (logits_L - official_logits).abs().max().item()
is_exact_match = torch.allclose(logits_L, official_logits, atol=1e-5)

print("\n--- SANITY CHECK RESULT ---")
print(f"Max absolute difference between logits_L and model(...).logits: {diff:.2e}")
print(f"Are all values numerically equal within tolerance? {is_exact_match}")
assert is_exact_match, "ERROR: Layer L logits do not match official model logits!"
print("✅ Sanity check PASSED successfully!")

Prompt: 'The capital of France is'
Token IDs shape: torch.Size([1, 5])
Extracted layers count: 13 (L + 1 = 13 for GPT-2)
Shape of each layer hidden state: torch.Size([1, 5, 768])
Computed lens logits count: 13
Shape of each lens logit: torch.Size([1, 5, 50257]) (B, T, V)

--- SANITY CHECK RESULT ---
Max absolute difference between logits_L and model(...).logits: 0.00e+00
Are all values numerically equal within tolerance? True
✅ Sanity check PASSED successfully!


## 6. Visualizing Layer-by-Layer Prediction Evolution for GPT-2

Inspect the top-1 predicted next token at the last sequence position ($t = -1$) as representation depth increases.

In [6]:
print(f"Layer-wise next-token prediction evolution for '{prompt}':\n")
print(f"{'Layer':<8} | {'Predicted Top-1 Token':<25} | {'Logit':<10}")
print("-" * 50)

for l, logits_l in enumerate(lens_logits):
    last_pos_logits = logits_l[0, -1, :]
    top1_id = last_pos_logits.argmax().item()
    top1_token = tokenizer.decode([top1_id])
    top1_val = last_pos_logits[top1_id].item()
    print(f"Layer {l:<2}   | {repr(top1_token):<25} | {top1_val:.2f}")

Layer-wise next-token prediction evolution for 'The capital of France is':

Layer    | Predicted Top-1 Token     | Logit     
--------------------------------------------------
Layer 0    | ' destro'                 | 87.56
Layer 1    | ' not'                    | 5.53
Layer 2    | ' not'                    | 5.87
Layer 3    | ' now'                    | 4.58
Layer 4    | ' now'                    | 4.79
Layer 5    | ' now'                    | 1.57
Layer 6    | ' now'                    | -0.57
Layer 7    | ' now'                    | -3.13
Layer 8    | ' now'                    | -7.24
Layer 9    | ' now'                    | -12.71
Layer 10   | ' France'                 | -17.24
Layer 11   | ' France'                 | -55.42
Layer 12   | ' the'                    | -101.04


### 6.1 Top-5 Candidate Tokens for GPT-2 (Layer 12)
Inspect the probability distribution over the top 5 next-token candidates at the final layer.

In [8]:
# Inspect top-5 candidate tokens predicted by GPT-2 at the final layer (Layer 12)
last_logits = lens_logits[-1][0, -1, :]
probs = torch.softmax(last_logits, dim=-1)
top5 = torch.topk(probs, k=5)

print(f"Top 5 candidates for GPT-2 at Layer 12 for '{prompt}':")
for rank, (token_id, prob) in enumerate(zip(top5.indices, top5.values), 1):
    token_str = tokenizer.decode([token_id.item()])
    print(f"  #{rank}: {repr(token_str):<15} (prob: {prob.item() * 100:.2f}%)")

Top 5 candidates for GPT-2 at Layer 12:
  #1: ' the'          (prob: 8.79%)
  #2: ' now'          (prob: 4.55%)
  #3: ' a'            (prob: 3.77%)
  #4: ' Paris'        (prob: 3.66%)
  #5: ' France'       (prob: 3.47%)


## 7. Verification with Italian Decoder: GePpeTto

As specified in issue `[L1]`, the lens must operate identically on `LorenzoDeMattei/GePpeTto`.

In [18]:
geppetto_spec = get_model_spec("geppetto")
print("GePpeTto spec:", geppetto_spec)

model_it, tokenizer_it, metadata_it = load_model(
    checkpoint=geppetto_spec["hf_id"],
    kind=geppetto_spec["kind"],
)

prompt_it = "La capitale d'Italia è"
inputs_it = tokenizer_it(prompt_it, return_tensors="pt")

hidden_states_it = run_model(model_it, inputs_it["input_ids"], inputs_it["attention_mask"])
lens_logits_it = logit_lens(model_it, hidden_states_it)

# Sanity check on GePpeTto
with torch.no_grad():
    official_logits_it = model_it(
        inputs_it["input_ids"].to(model_it.device),
        attention_mask=inputs_it["attention_mask"].to(model_it.device)
    ).logits

diff_it = (lens_logits_it[-1] - official_logits_it).abs().max().item()
assert torch.allclose(lens_logits_it[-1], official_logits_it, atol=1e-5), "Sanity check failed on GePpeTto!"
print(f"✅ Sanity check on GePpeTto PASSED! (max diff: {diff_it:.2e})")

# Prediction inspection for GePpeTto
print(f"\nLayer-wise prediction evolution for '{prompt_it}':")
for l, logits_l in enumerate(lens_logits_it):
    top1_id = logits_l[0, -1, :].argmax().item()
    print(f"Layer {l:<2}: {repr(tokenizer_it.decode([top1_id]))}")

GePpeTto spec: {'key': 'geppetto', 'hf_id': 'LorenzoDeMattei/GePpeTto', 'kind': 'decoder', 'lang': 'it', 'num_layers': 12, 'hidden_size': 768, 'dtype': 'float32', 'revision': 'main'}


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7158.51it/s]


✅ Sanity check on GePpeTto PASSED! (max diff: 0.00e+00)

Layer-wise prediction evolution for 'La capitale d'Italia è':
Layer 0 : ' è'
Layer 1 : ' stata'
Layer 2 : ' stata'
Layer 3 : ' stata'
Layer 4 : ' stata'
Layer 5 : ' stata'
Layer 6 : ' stata'
Layer 7 : ' stata'
Layer 8 : ' stata'
Layer 9 : ' stata'
Layer 10: ' stata'
Layer 11: ' la'
Layer 12: ' la'


### 7.1 Top-5 Candidate Tokens for GePpeTto (Layer 12)
Inspect the probability distribution over the top 5 next-token candidates for the Italian model.

In [19]:
# Inspect top-5 candidate tokens predicted by GePpeTto at the final layer (Layer 12)
last_logits_it = lens_logits_it[-1][0, -1, :]
probs_it = torch.softmax(last_logits_it, dim=-1)
top5_it = torch.topk(probs_it, k=5)

print(f"Top 5 candidates for GePpeTto at Layer 12 for '{prompt_it}':")
for rank, (token_id, prob) in enumerate(zip(top5_it.indices, top5_it.values), 1):
    token_str = tokenizer_it.decode([token_id.item()])
    print(f"  #{rank}: {repr(token_str):<15} (prob: {prob.item() * 100:.2f}%)")

Top 5 candidates for GePpeTto at Layer 12 for 'La capitale d'Italia è':
  #1: ' la'           (prob: 9.43%)
  #2: ' il'           (prob: 3.39%)
  #3: ' stata'        (prob: 3.00%)
  #4: ' Milano'       (prob: 2.88%)
  #5: ' l'            (prob: 2.13%)
